# 01. Chuẩn bị Dữ liệu (Data Preparation)

Quy trình này nhằm mục đích tải và tiền xử lý các tập dữ liệu, đồng thời thực hiện chia tập dữ liệu huấn luyện và kiểm thử theo thời gian để đảm bảo đánh giá chuẩn xác cho hệ thống gợi ý.

---

### Phân tích Quyết định Thiết kế:
*   **Tại sao chọn phân chia theo thời gian (Time-Based / Leave-One-Out)?**
    *   Trong môi trường thực tế, hệ thống gợi ý luôn nhận dữ liệu lịch sử và phải đưa ra đề xuất cho hành động tiếp theo ở tương lai. Chia tập train/test ngẫu nhiên (Random Split) sẽ gây ra lỗi **rò rỉ dữ liệu (Data Leakage)** khi lấy tương tác ở tương lai để dự đoán quá khứ, làm ảo tưởng hiệu năng thực tế. Phương pháp **Leave-One-Out (LOO)** lấy tương tác cuối cùng của mỗi user làm Test mô phỏng chính xác nhất quy trình vận hành này.
*   **Tại sao không chọn K-Fold Cross Validation ngẫu nhiên?**
    *   K-Fold ngẫu nhiên chia cắt hoàn toàn yếu tố thời gian và phá vỡ cấu trúc chuỗi hành vi của người dùng, dẫn đến kết quả đánh giá không thực tế trong hệ gợi ý.


In [1]:
import os
import sys
import pandas as pd
import numpy as np
import pickle

# Thêm đường dẫn cha để import recsys_utils
sys.path.append(os.path.abspath('..'))
from recsys_utils import split_data_implicit_leave_one_out

# Cấu hình đường dẫn dữ liệu
data_dir = os.path.join("..", "..", "data")
simulator_dir = os.path.join(data_dir, "simulator")

# 1. Load các tệp dữ liệu simulator
users_df = pd.read_csv(os.path.join(simulator_dir, "sim_users.csv"))
ratings_df = pd.read_csv(os.path.join(simulator_dir, "sim_ratings.csv"))
clicks_df = pd.read_csv(os.path.join(simulator_dir, "sim_click_events.csv"))
movies_df = pd.read_csv(os.path.join(data_dir, "crawler", "movies_crawled.csv"))

print(f"Loaded {len(users_df)} users.")
print(f"Loaded {len(ratings_df)} ratings.")
print(f"Loaded {len(clicks_df)} click events.")
print(f"Loaded {len(movies_df)} movies in catalog.")


Loaded 200 users.
Loaded 2266 ratings.
Loaded 48719 click events.
Loaded 10000 movies in catalog.


In [2]:
# 2. Phân chia tập dữ liệu theo Leave-One-Out (Implicit Feedback)
# Sử dụng ratings làm base và map với clicks để sinh test set LOO
# Để đồng nhất, ta dùng hàm split_data_implicit_leave_one_out trên ratings_df
train_ratings, test_data, user_interacted_items = split_data_implicit_leave_one_out(
    ratings_df, user_col='userId', item_col='movieId', timestamp_col='timestamp', seed=42
)

# Cập nhật user_interacted_items để gộp thêm các phim người dùng đã click
# Nhằm tránh lỗi chọn mẫu âm trúng các phim người dùng đã click trong clicks_df
click_interacted = clicks_df.groupby('userId')['movieId'].apply(set).to_dict()
for u, clicked_set in click_interacted.items():
    if u in user_interacted_items:
        user_interacted_items[u] = user_interacted_items[u].union(clicked_set)
    else:
        user_interacted_items[u] = clicked_set

# Lưu lại các tập dữ liệu đã chia để các notebook sau sử dụng
os.makedirs("processed_data", exist_ok=True)
train_ratings.to_csv("processed_data/train_ratings.csv", index=False)

with open("processed_data/test_data.pkl", "wb") as f:
    pickle.dump(test_data, f)
    
with open("processed_data/user_interacted_items.pkl", "wb") as f:
    pickle.dump(user_interacted_items, f)

print(f"Chia dữ liệu thành công! Train ratings: {len(train_ratings)} | Test instances (users): {len(test_data)}")


Chia dữ liệu thành công! Train ratings: 2066 | Test instances (users): 200
